In [58]:
""" 
Created on 16-08-2026

@author: B.A. Sturre

feature engineering for SaaS dataset 
"""

import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
import torch
import torch.nn as nn
from sklearn.metrics import classification_report, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

# matplotlib setup
plt.rcParams.update({
    "text.usetex": False,
    "font.family": "serif",
    "font.serif": ["DejaVu Serif"],
    "mathtext.fontset": "dejavuserif"
})

### reading data
as in `EDA.ipynb`

In [59]:
# reading in the data 
accounts = pd.read_csv('data/ravenstack_accounts.csv')
churns = pd.read_csv('data/ravenstack_churn_events.csv')
feature = pd.read_csv('data/ravenstack_feature_usage.csv')
subs = pd.read_csv('data/ravenstack_subscriptions.csv')
support = pd.read_csv('data/ravenstack_support_tickets.csv')

# aggregating feature usage by subscription id
feature_agg = feature.groupby('subscription_id').agg(
    total_usage_count=('usage_count', 'sum'),
    total_usage_duration_min=('usage_duration_secs', lambda x: x.sum() / 60),
    total_error_count=('error_count', 'sum')
).reset_index()

# aggregating support aggregation by account id 
support_agg = support.groupby('account_id').agg(
    total_tickets=('ticket_id', 'count'),
    avg_resolution_hours=('resolution_time_hours', 'mean'),
    avg_satisfaction_score=('satisfaction_score', 'mean'),
    total_escalations=('escalation_flag', 'sum')
).reset_index()

# merging datasets
df = subs.merge(feature_agg, on='subscription_id', how='left')
df = df.merge(accounts[['account_id', 'industry', 'country', 'referral_source']], on='account_id', how='left')
df = df.merge(support_agg, on='account_id', how='left')

# target creation
churn_ids = churns['subscription_id'].unique() if 'subscription_id' in churns.columns else churns['account_id'].unique()
churn_col = 'subscription_id' if 'subscription_id' in churns.columns else 'account_id'
df['is_churned'] = df[churn_col].isin(churn_ids).astype(int)

# fill NaN values for accounts with zero usage
df['total_usage_count'] = df['total_usage_count'].fillna(0)
df['total_usage_duration_min'] = df['total_usage_duration_min'].fillna(0)
df['total_error_count'] = df['total_error_count'].fillna(0)
df['total_tickets'] = df['total_tickets'].fillna(0)
df['total_escalations'] = df['total_escalations'].fillna(0)
df['avg_resolution_hours'] = df['avg_resolution_hours'].fillna(0)
df['avg_satisfaction_score'] = df['avg_satisfaction_score'].fillna(df['avg_satisfaction_score'].median())

# create account size categories
df['account_size_category'] = pd.cut(
    df['seats'], 
    bins=[0, 5, 20, 50, np.inf], 
    labels=['Micro (1-5)', 'Small (6-20)', 'Medium (21-50)', 'Enterprise (50+)']
)

# calculating average feature usage count, usage durations, ratio of errors, support tickets, escalation rate, MRR per seat
df['usage_per_seat'] = df['total_usage_count'] / np.maximum(df['seats'], 1)
df['usage_duration_per_seat'] = df['total_usage_duration_min'] / np.maximum(df['seats'], 1)
df['error_rate'] = df['total_error_count'] / (df['total_usage_count'] + 1e-5)
df['tickets_per_seat'] = df['total_tickets'] / np.maximum(df['seats'], 1)
df['escalation_rate'] = df['total_escalations'] / (df['total_tickets'] + 1e-5)
df['mrr_per_seat'] = df['mrr_amount'] / np.maximum(df['seats'], 1)

# removing outliers 
def remove_outliers(df, columns, iqr_multiplier=3.0):
    df_clean = df.copy()
    for col in columns:
        valid_data = df_clean[col].dropna()
        q1 = valid_data.quantile(0.25)
        q3 = valid_data.quantile(0.75)
        iqr = q3 - q1
        
        upper_bound = q3 + (iqr_multiplier * iqr)
        lower_bound = q1 - (iqr_multiplier * iqr)
        
        outlier_mask = (df_clean[col] < lower_bound) | (df_clean[col] > upper_bound)
        df_clean = df_clean[~outlier_mask]
    return df_clean

# remove outliers
df_clean = remove_outliers(
    df, 
    columns=['mrr_amount', 'total_usage_count', 'total_tickets', 'avg_resolution_hours'], 
    iqr_multiplier=3.0
)

### Feature Engineering

In [60]:
# feature column names for churn prediction 
features = [
    'seats', 'mrr_amount', 'total_usage_count', 'total_usage_duration_min',
    'total_error_count', 'total_tickets', 'avg_resolution_hours',
    'avg_satisfaction_score', 'total_escalations', 'industry', 'account_size_category',
    'usage_per_seat', 'usage_duration_per_seat', 'error_rate', 
    'tickets_per_seat', 'escalation_rate', 'mrr_per_seat'
]

# binary indicator columns
X_df = pd.get_dummies(df_clean[features], drop_first=True)

# extract binary target variable
y_series = df_clean['is_churned'].values

# split data into 80% train+val and 20% test
X_temp, X_test_raw, y_temp, y_test_raw = train_test_split(
    X_df.values.astype(np.float32),
    y_series.astype(np.float32),
    test_size=0.2,
    random_state=42,
    stratify=y_series,
)

# splitting into training & validation 
X_train_raw, X_val_raw, y_train_raw, y_val_raw = train_test_split(
    X_temp, y_temp, test_size=0.2, random_state=42, stratify=y_temp
)

# standardize features
scaler = StandardScaler()

# compute mean and std on training set and transform training features
X_train_scaled = scaler.fit_transform(X_train_raw)

# Transform validation features using parameters fitted strictly on training data
X_val_scaled = scaler.transform(X_val_raw)

# Transform test features using parameters fitted strictly on training data
X_test_scaled = scaler.transform(X_test_raw)

In [61]:
# convert scaled feature arrays into PyTorch Tensors
X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32)
X_val_t = torch.tensor(X_val_scaled, dtype=torch.float32)
X_test_t = torch.tensor(X_test_scaled, dtype=torch.float32)

# convert label vectors into 2D column Tensors with shape (N, 1)
y_train_t = torch.tensor(y_train_raw, dtype=torch.float32).unsqueeze(1)
y_val_t = torch.tensor(y_val_raw, dtype=torch.float32).unsqueeze(1)
y_test_t = torch.tensor(y_test_raw, dtype=torch.float32).unsqueeze(1)

# package training features and labels into unified TensorDataset
train_dataset = TensorDataset(X_train_t, y_train_t)

# wrap dataset in DataLoader for mini-batch generation with shuffling
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

In [62]:
# neural network classifier
class DeepChurnClassifier(nn.Module):
    def __init__(self, input_dim):
        super(DeepChurnClassifier, self).__init__()
        # layer progression
        self.network = nn.Sequential(
            # input layer input -> 128 
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            
            # hidden layer 128 -> 64
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2),
            
            # hidden layer 64 -> 32
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.1),
            
            # output layer outputting single layer
            nn.Linear(32, 1)
        )
        
    def forward(self, x):
        return self.network(x)

# total input feature count
input_dim = X_train_t.shape[1]

# make model
model = DeepChurnClassifier(input_dim)

# compute pos_weight factor (Non-Churned count / Churned count) for BCE loss calculation
pos_weight = torch.tensor([(len(y_train_raw) - sum(y_train_raw)) / sum(y_train_raw)])

# loss function
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# AdamW optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

# larning rate scheduler: reduces LR if validation loss fails to decrease
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5
)

In [63]:
#initialization
epochs = 100
patience = 12
best_val_loss = float('inf')
patience_counter = 0
best_model_weights = None

for epoch in range(epochs):
    # set model to training mode
    model.train()
    train_loss = 0.0
    
    # mini-batch execution loop
    for batch_X, batch_y in train_loader:
        # zero out accumulated parameter gradients
        optimizer.zero_grad()
        # compute forward pass predictions
        predictions = model(batch_X)
        # calculate loss on current batch
        loss = criterion(predictions, batch_y)
        # calculate gradients via backpropagation
        loss.backward()
        # update weights using optimizer
        optimizer.step()
        # accumulate total loss across batch
        train_loss += loss.item() * batch_X.size(0)
        
    train_loss = train_loss / len(train_loader.dataset)
    
    # set model to evaluation mode for validation step
    model.eval()
    with torch.no_grad():
        val_logits = model(X_val_t)
        val_loss = criterion(val_logits, y_val_t).item()
        
    # step learning rate scheduler based on validation loss performance
    scheduler.step(val_loss)
    
    # early stopping and checkpointing logic based on minimum validation loss
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        best_model_weights = copy.deepcopy(model.state_dict())
    else:
        patience_counter += 1
        
    if patience_counter >= patience:
        print(f"Early stopping triggered at epoch {epoch + 1}")
        break

# restore best observed model parameters before evaluation
if best_model_weights is not None:
    model.load_state_dict(best_model_weights)

Early stopping triggered at epoch 79


In [64]:
# model in evaluation mode
model.eval()

with torch.no_grad():
    # predict validation set raw logits
    val_logits = model(X_val_t)
    # apply Sigmoid activation to convert logits into probabilities
    val_probs = torch.sigmoid(val_logits).numpy().flatten()

# search candidate thresholds from 0.05 to 0.95 in increments of 0.01 to maximize validation F1-score
best_threshold = 0.5
best_val_f1 = 0.0
candidate_thresholds = np.arange(0.05, 0.95, 0.01)

for thresh in candidate_thresholds:
    # convert probabilities to binary predictions using current trial threshold
    thresh_preds = (val_probs >= thresh).astype(int)
    # calculate F1-score for churn positive class
    f1 = f1_score(y_val_raw, thresh_preds, zero_division=0)
    # store threshold yielding the highest validation F1-score
    if f1 > best_val_f1:
        best_val_f1 = f1
        best_threshold = thresh

print(f"Optimal decision threshold tuned on Validation Set: {best_threshold:.2f} (Val F1: {best_val_f1:.4f})")

Optimal decision threshold tuned on Validation Set: 0.22 (Val F1: 0.8665)


In [ ]:
with torch.no_grad():
    # pass test tensor through model to obtain logits
    test_logits = model(X_test_t)
    # convert test logits to probabilities
    test_probs = torch.sigmoid(test_logits).numpy().flatten()
    # apply optimal decision threshold tuned on validation set to generate final binary predictions
    test_preds = (test_probs >= best_threshold).astype(int)

# compute final ROC-AUC score on held-out test probabilities
test_auc = roc_auc_score(y_test_raw, test_probs)

print(f"ROC-AUC:{test_auc:.4f}")
print(f"Applied decision threshold: {best_threshold:.2f}")
print(classification_report(y_test_raw, test_preds))

ROC-AUC:0.8023
Applied Decision Threshold: 0.22
              precision    recall  f1-score   support

         0.0       0.79      0.33      0.46       282
         1.0       0.77      0.96      0.86       679

    accuracy                           0.78       961
   macro avg       0.78      0.64      0.66       961
weighted avg       0.78      0.78      0.74       961



## Suggested Projects

- Churn prediction using subscriptions + support data

- Feature adoption tracking during beta phases

- Support workload forecasting

- Revenue cohort analysis by referral channel

- Plan tier upgrade funnel by industry

- Latency analysis by seat count and plan tier